# The deeperfly pipeline, step by step

This notebook reproduces what **`deeperfly run`** does, but one module at a time,
so every intermediate output is visible and plotted. We run on the synchronized
7-camera recording in `examples/data/` (one MP4 per camera, `camera_0.mp4` …
`camera_6.mp4`).

`deeperfly run` is driven entirely by **one config**. The packaged default
(`Config.default()`, the `data/default_config.toml` used when you pass no `-c`)
already points its footage sources at this recording, so we load it and let it
wire everything together:

1. **the detection plan** (`Config.detection_plan`) — the config-driven
   description of *what to detect*: named footage **sources**, reusable
   **preprocessors** (mirror/crop/resize), detector **models**, and the
   **pathways** (`source → preprocessor → model`) that run them. A separate
   `[pose2d.output_points]` table says where each pathway's output channels land in the
   `(view, skeleton-point)` grid.
2. **the camera rig** (`Config.camera_group`) and **skeleton** (`Config.skeleton`).
3. **`pose2d` models + `inference.detect_sequence`** — preprocess → heatmaps →
   map each pathway's peaks back into its view → a `(V, T, 38, 2)` 2D pose.
4. **`pipeline.run_from_points2d`** — bundle adjustment →
   robust triangulation (RANSAC multi-view consensus) → a saved
   **`results.PoseResult`**.

Two things to note about the design: there is **no separate visibility mask and
no special-cased front camera**. A `(view, point)` that no pathway writes simply
stays `NaN` — that *is* the visibility mask. The front camera is just one source
feeding **two** pathways (one mirrored), one reading its right legs and one its
left. We unfold all of this below and visualize each stage with matplotlib; the
final cells call `run_from_points2d` to show it is the same thing in one call.

## Setup

Import the precise pieces the CLI composes, and load the packaged default config.
`N_FRAMES` controls how much of the 64-frame recording we process — small enough
to run quickly, large enough for bundle adjustment to have something to chew on.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The exact modules `deeperfly run` composes:
from deeperfly import Config, io
from deeperfly.pipeline import (
    bundle_adjust_cameras,
    reconstruct_ransac,
    run_from_points2d,
)
from deeperfly.pose2d import inference
from deeperfly.pose2d.pathways import (
    normalized_peaks_to_original_pixels,
    route_channels_to_points_in_views,
)
from deeperfly.pose2d.stream import load_models
from deeperfly.results import PoseResult
from deeperfly.triangulation import reprojection_error, triangulate
from deeperfly.visualization._palette import point_colors_rgb

# Render plots inline. deeperfly's own rendering is headless OpenCV (no matplotlib);
# the plots below are drawn with matplotlib directly for the walkthrough.
%matplotlib inline

# Locate the repo root (so the notebook works regardless of the launch dir).
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
DATA_DIR = REPO / "examples" / "data"

# The packaged default config -- exactly what `deeperfly run` uses with no `-c`.
# It is self-contained (detection plan + cameras + skeleton + pipeline) and its
# footage sources already map onto camera_0.mp4 .. camera_6.mp4 in examples/data.
config = Config.default()

N_FRAMES = 60  # frames to process (each example video has 64)
FPS = 100.0  # `deeperfly run` detects this from the videos; we set it directly

patterns = config.source_patterns()  # source name -> footage glob
first_file = DATA_DIR / next(iter(patterns.values()))
n_avail = io.open_reader(first_file).count()
print(f"repo:    {REPO}")
print(f"data:    {DATA_DIR}  ({n_avail} frames/camera available)")
print(f"sources: {patterns}")

## Step 1 — the camera rig and the skeleton

`Config.detection_plan` parses the plan and resolves the **views** (the geometric
cameras under `[cameras.*]`) — the `V` axis of every points array. `Config.skeleton`
is the rig-independent description of *what* is tracked: 38 points (left `0..18`,
right `19..37`), grouped into limbs and joined by bones. `Config.camera_group`
builds the rig geometry from the same TOML.

The config leaves `principal_point_px` unset, so we hand `camera_group` each
camera's `(H, W)` and it places the principal point at the image center — exactly
what `deeperfly run` does from the footage (these frames are 960×480). The
bundle-adjustment options come straight off `config.bundle_adjustment`: the
leg-only `points_to_use` (point names) that drive bundle adjustment and the
`fixed`/`shared` parameters that anchor the world gauge.

In [ ]:
plan = config.detection_plan()
skeleton = config.skeleton()
view_names = plan.view_names  # the V axis order, from [cameras.*]

# The frames are 960x480 and the config leaves principal_point_px unset, so pass
# each camera's (H, W) and camera_group infers the principal point as the center.
probe = np.asarray(io.open_reader(first_file)[:1])[0]
H, W = probe.shape[:2]
image_sizes = {name: (H, W) for name in view_names}
cameras = config.camera_group(image_sizes=image_sizes)

# Bundle-adjustment options, mapped to `bundle_adjust_cameras`'s kwargs the same way
# `deeperfly run` does: the leg-only `points_to_use` (point *names*) resolved to
# the `ba_keypoints` indices that drive bundle adjustment, the `fixed`/`shared` world
# gauge, and the scipy least_squares kwargs (max_nfev, loss).
ba = config.bundle_adjustment
bundle_adjust_kwargs = {"fixed": ba.fixed, "shared": ba.shared, **ba.least_squares}
if ba.points_to_use is not None:
    point_index = {name: i for i, name in enumerate(skeleton.point_names)}
    bundle_adjust_kwargs["ba_keypoints"] = [point_index[n] for n in ba.points_to_use]

print(
    f"skeleton: {skeleton.name!r}  {skeleton.n_points} points, "
    f"{len(skeleton.bones)} bones, {skeleton.n_limbs} limbs"
)
print(f"limbs: {skeleton.limb_names}")
print(f"views: {view_names}")
print(f"frame size (H, W) = {(H, W)};  principal point -> {[(W - 1) / 2, (H - 1) / 2]}")
for c in cameras:
    print(f"  {c.name:>3}: center={np.round(c.position, 1)}  focal={c.intr[0]:.0f}px")

In [ ]:
from deeperfly.cameras import Camera


def plot_camera(camera: Camera, ax, length=None, **kwargs):
    "Draw a camera as an RGB axis triad at its world center"
    if length is None:
        length = np.linalg.norm(camera.tvec) * 0.2
    for axis, color in zip(camera.rmat, "rgb"):
        ax.plot(
            *zip(camera.position, camera.position + axis * length),
            color=color,
            **kwargs,
        )

In [ ]:
# The 7 cameras orbit the world origin (where the fly sits). Plot their centers.
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
for c in cameras:
    plot_camera(c, ax, length=10, alpha=0.8)
    ax.text(
        *c.position + np.array([0, 0, 10]),
        s=c.name,
        color="k",
        fontsize=9,
        ha="center",
        va="center",
    )
ax.scatter([0], [0], [0], color="k", marker="x", s=70)
ax.set_title("Camera rig: centers orbit the world origin")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_aspect("equal")
ax.view_init(elev=30, azim=-60)
ax.set_zticks([0])
fig.tight_layout()
plt.show()

## Step 2 — the detection plan

The plan keeps four counts independent rather than fusing them at "one per
camera":

- **sources** — named footage globs, each decoded once.
- **preprocessors** — reusable frame-op pipelines (here just `noflip` and a
  left-right `fliplr` mirror).
- **models** — detector networks (here one stacked-hourglass `deepfly2d`).
- **pathways** — each a named `source → preprocessor → model` inference run.

Where a pathway's 19 output channels land is declared separately in the
`[pose2d.output_points.<view>]` tables, resolved into each pathway's `(out_channel,
view, point)` mapping. The **front source `vid_f` feeds two pathways**: `f_noflip`
reads the right distal joints and `f_fliplr` (mirrored) reads the left — so the one
front image observes joints on **both** body sides. That front-camera bridge is
what lets bundle adjustment tie the otherwise-disjoint left and right cameras into a
single world frame (see Step 9).

In [ ]:
from collections import Counter

print("views:", plan.view_names)
print("\nsources:")
for s in plan.sources:
    print(f"  {s.name:>7}  <- {s.pattern}")
print("\npreprocessors:")
for name, t in plan.preprocessors.items():
    print(f"  {name:>7}: {t.to_json()}")
print("\nmodels:")
for name, spec in plan.models.items():
    print(
        f"  {name}: class={spec.cls!r} input_size={spec.input_size} "
        f"n_out_channels={spec.n_out_channels}"
    )

print("\npathways (source -> preprocessor -> model -> views it fills):")
for pw in plan.pathways:
    views = sorted({plan.view_names[v] for v in pw.mapping[:, 1]})
    print(
        f"  {pw.name:>10}: {pw.source:>7} --{pw.preprocessor}--> {pw.model}"
        f"  -> {views}  ({len(pw.mapping)} points)"
    )

src_counts = Counter(pw.source for pw in plan.pathways)
multi = {s: c for s, c in src_counts.items() if c > 1}
print(f"\nsource feeding >1 pathway (the front bridge): {multi}")

## Step 3 — load the detector model(s)

`load_models` loads every model the plan references into a `name → LoadedModel`
dict (downloading the cached DeepFly2D checkpoint on first use). A `LoadedModel`
owns its input contract — resize to its `input_size` (256×512) and subtract its
training `mean` — and runs the forward/decode on the GPU when one is available.
Every pathway forwards through the model named in its `model` key.

In [ ]:
models = load_models(plan)
for name, m in models.items():
    print(
        f"{name}: device={m.device()}  input_size={m.input_size}  "
        f"channels={m.n_out_channels}  mean={m.spec.mean}"
    )

## Step 4 — load synchronized frames

`deeperfly run` decodes each **source** once. We decode the first `N_FRAMES` of
each source's video with `io.open_reader(...)[:N_FRAMES]` — the same reader the
CLI uses — into a `windows` dict mapping each source name to a `(T, H, W, 3)`
array. For frame `t`, the synchronized views are `windows[src][t]` across sources.

In [ ]:
# Decode the first N_FRAMES of each source's video into a (T, H, W, 3) window.
windows = {
    name: np.asarray(io.open_reader(DATA_DIR / pat)[:N_FRAMES])
    for name, pat in patterns.items()
}
print("windows:", {k: v.shape for k, v in windows.items()})

# Each view's footage comes from the source feeding it (plan.view_sources()).
view_src = plan.view_sources()
fig, axes = plt.subplots(2, 4, figsize=(12, 4))
for v, ax in enumerate(axes.ravel()):
    if v < len(view_names):
        name = view_names[v]
        ax.imshow(windows[view_src[name]][0], cmap="gray")
        ax.set_title(f"{name}  <- {view_src[name]}", fontsize=8)
    ax.axis("off")
fig.suptitle("Synchronized raw frame 0 across the 7 views")
fig.tight_layout()
plt.show()

## Step 5 — preprocessing per pathway

Each pathway orients its source frame with its **preprocessor** (`fliplr` mirrors
far-side cameras so the fly faces the trained orientation; `noflip` is the
identity), then the **model** resizes to its 256×512 input and subtracts the
training mean. We run this per pathway, so the front source `vid_f` appears twice:
once un-flipped (its right-leg pathway) and once mirrored (its left-leg pathway).
Below, each pathway before and after preprocessing — note how the mirror makes
every view face the same way.

In [ ]:
def pathway_label(pw):
    "Short name for a pathway, e.g. 'f_fliplr (vid_f, fliplr)'."
    return f"{pw.name} ({pw.source}, {pw.preprocessor})"


# Preprocess one frame per PATHWAY (the front source -> two pathways, one mirrored).
preprocessed_disp = []
for pw in plan.pathways:
    model = models[pw.model]
    oriented = pw.transform.apply(windows[pw.source][:1])  # (1, H, W, 3)
    prepared = model.prepare(oriented)  # (1, 3, Hh, Ww), on the model's device
    img = prepared[0].float().cpu().numpy().transpose(1, 2, 0) + model.spec.mean
    preprocessed_disp.append(np.clip(img, 0, 1))

n_pw = len(plan.pathways)
fig, axes = plt.subplots(4, 4, figsize=(12, 8))
axs_raw = axes[0::2].ravel()
axs_pre = axes[1::2].ravel()
for i, (ax_raw, ax_pre) in enumerate(zip(axs_raw, axs_pre)):
    if i < n_pw:
        pw = plan.pathways[i]
        ax_raw.set_title(f"{pathway_label(pw)}, raw", fontsize=8)
        ax_raw.imshow(windows[pw.source][0], cmap="gray")
        ax_pre.set_title(f"{pathway_label(pw)}, preprocessed", fontsize=8)
        ax_pre.imshow(preprocessed_disp[i], cmap="gray")
    ax_raw.axis("off")
    ax_pre.axis("off")
fig.suptitle(
    "preprocess() per pathway: the front source appears twice (right + flipped left)"
)
fig.tight_layout()
plt.show()

## Step 6 — heatmaps

Each model's forward pass returns one heatmap per output channel (19 single-side
channels) at the network's output resolution. The peak of each heatmap is the
predicted joint location. We run every pathway's frame-0 input through its model
and overlay the heatmaps onto each preprocessed pathway below.

In [ ]:
heatmaps = []
for pw in plan.pathways:
    model = models[pw.model]
    oriented = pw.transform.apply(windows[pw.source][:1])
    hm = model.predict_heatmaps(model.prepare(oriented))[0]  # (J, Hh, Ww)
    heatmaps.append(hm)
heatmaps = np.stack(heatmaps)  # (P, J, Hh, Ww)
print("heatmaps:", heatmaps.shape, "  (pathways, channels, Hh, Ww)")

In [ ]:
def channel_colors(pathway, skeleton, n_out):
    """RGB per model output channel, by the skeleton point each maps to.

    A pathway's mapping is (out_channel, view, point) triples; color channel i
    by its target point's limb color (gray for any channel the pathway drops).
    """
    pts = point_colors_rgb(skeleton)
    out = np.full((n_out, 3), 0.6)  # unmapped channels -> gray
    for i, _v, p in pathway.mapping:
        out[i] = pts[p]
    return out

In [ ]:
n_out = models[plan.pathways[0].model].n_out_channels
ext = [0, preprocessed_disp[0].shape[1], preprocessed_disp[0].shape[0], 0]
im = np.empty((heatmaps.shape[2], heatmaps.shape[3], 4))
fig, axes = plt.subplots(2, 4, figsize=(12, 4))
for i, ax in enumerate(axes.ravel()):
    if i < n_pw:
        pw = plan.pathways[i]
        colors = channel_colors(pw, skeleton, n_out)
        ax.set_facecolor("k")
        ax.imshow(preprocessed_disp[i], cmap="gray", alpha=0.5)
        ax.set_title(pathway_label(pw), fontsize=8)
        for j in range(heatmaps.shape[1]):
            im[..., :3] = colors[j]
            im[..., 3] = np.clip(heatmaps[i, j] * 2, 0, 1)
            ax.imshow(im, alpha=0.5, extent=ext)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("Stacked-hourglass heatmaps (one set per pathway)")
fig.tight_layout()
plt.show()

## Step 7 — decode peaks, map back to the view, and scatter into the skeleton

`heatmap_to_points` takes the (sub-pixel-refined) peak of each heatmap — a
normalized `(x, y)` plus a confidence. `normalized_peaks_to_original_pixels` then
inverts the pathway's preprocessing (undoing the model's resize and any
mirror/crop) to put each peak back into raw **view** pixels, and
`route_channels_to_points_in_views` writes channel `i` into `(view, point)` per
the pathway's mapping. Because the front source's two pathways both target view
`f`, they land on the **same row**, filling both halves. In the plot below the
front view now carries *both* leg sets; every other view carries one.

In [ ]:
out_pts = np.full((plan.n_views, plan.n_points, 2), np.nan)
out_conf = np.zeros((plan.n_views, plan.n_points))
for i, pw in enumerate(plan.pathways):
    model = models[pw.model]
    pn, cc = inference.heatmap_to_points(heatmaps[i])  # (J, 2) in [0,1], (J,)
    src_hw = windows[pw.source].shape[1:3]  # (H, W) of the raw source frame
    raw_xy = normalized_peaks_to_original_pixels(
        pn, pw.transform, model.input_size, src_hw
    )
    route_channels_to_points_in_views(raw_xy, cc, pw.mapping, out_pts, out_conf)

front = view_names.index("f")
print("assembled 2D points:", out_pts.shape)
print(
    f"front view fills left half: {np.isfinite(out_pts[front, :19]).any()}, "
    f"right half: {np.isfinite(out_pts[front, 19:]).any()}"
)

fig, axes = plt.subplots(2, 4, figsize=(12, 4))
colors = point_colors_rgb(skeleton)
for v, ax in enumerate(axes.ravel()):
    if v < len(view_names):
        ax.set_facecolor("k")
        ax.imshow(windows[view_src[view_names[v]]][0], cmap="gray")
        extra = " - both sides" if v == front else ""
        ax.set_title(f"{view_names[v]}{extra}", fontsize=8)
        for a, b in skeleton.bones:
            ax.plot(out_pts[v, [a, b], 0], out_pts[v, [a, b], 1], color=colors[a])
    ax.axis("off")
fig.suptitle("Assembled 2D skeleton (the front view carries both leg sets)")
fig.tight_layout()
plt.show()

## Step 8 — detect the whole sequence

`inference.detect_sequence` repeats steps 5–7 for every frame in one fully-batched
pass over the `windows`, giving the full 2D result: `pts2d` of shape
`(V, T, 38, 2)` in pixels and `conf` of shape `(V, T, 38)`. This is the array
`deeperfly run` feeds into the geometry pipeline (`deeperfly.detect_2d` wraps this
with streaming decode so memory stays bounded over long recordings).

In [ ]:
pts2d, conf = inference.detect_sequence(plan, models, windows)
print("pts2d:", pts2d.shape, "  conf:", conf.shape)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4))
im = axa.imshow(np.nanmean(conf, axis=1), aspect="auto", cmap="viridis", vmin=0, vmax=1)
axa.set_yticks(range(len(view_names)))
axa.set_yticklabels(view_names)
axa.set_xlabel("joint index  (0..18 left, 19..37 right)")
axa.set_title("Mean detector confidence per (view, joint)")
fig.colorbar(im, ax=axa, fraction=0.046)

j, v = 23, view_names.index("rm")  # rf claw seen by camera rm
axb.plot(pts2d[v, :, j, 0], label="x")
axb.plot(pts2d[v, :, j, 1], label="y")
axb.set_xlabel("frame")
axb.set_ylabel("pixel")
axb.legend()
axb.set_title(f"{view_names[v]}: 2D track of {skeleton.point_names[j]}")
fig.tight_layout()
plt.show()

## Step 9 — visibility is built into the plan

There is no separate visibility-masking step anymore. A `(view, point)` that no
pathway writes is simply left `NaN` by the scatter in Step 7, so the detector's
output is *already* masked. `plan.visibility_mask()` recovers which `(view, point)`
pairs any pathway fills, and it matches the finite entries of `pts2d` exactly. The
front (`f`) row is the only one with white cells on **both** halves — that
cross-side visibility is the bridge that co-registers the left and right cameras
during bundle adjustment.

In [ ]:
mask = plan.visibility_mask()  # (V, N) bool: which (view, point) any pathway fills
ever_seen = np.isfinite(pts2d).all(-1).any(axis=1)  # (V, N) finite somewhere in time
print(
    f"pts2d finite pattern matches the plan's visibility mask: {(ever_seen == mask).all()}"
)

fr = mask[front]
print(
    f"front view 'f' sees {int(fr.sum())} points spanning both sides "
    f"(left {int(fr[:19].sum())}, right {int(fr[19:].sum())})"
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(mask, aspect="auto", cmap="Greys_r", interpolation="nearest")
ax.set_yticks(range(len(view_names)))
ax.set_yticklabels(view_names)
ax.set_xlabel("joint index")
ax.set_title("Visibility mask (white = some pathway fills this (view, point))")
for i in range(mask.shape[1]):
    ax.axvline(i - 0.5, color="gray", lw=0.5)
ax.set_xticks(
    np.arange(mask.shape[1]),
    labels=skeleton.point_names,
    rotation=60,
    rotation_mode="anchor",
    ha="right",
)
ax.tick_params(axis="x", which="both", length=0)
fig.tight_layout()
plt.show()

## Step 10 — bundle adjustment

`pipeline.bundle_adjust_cameras` treats the **fly itself** as the bundle-adjustment target: it
flattens the frames into one point cloud and refines the cameras by bundle
adjustment, using detector confidences as per-observation weights, a robust loss,
and a soft bone-length prior. The detector's `pts2d` is already NaN where
unobserved, so it goes straight in. We compare reprojection error before vs after.

In [ ]:
# Reprojection error with the nominal (un-refined) cameras ...
pts3d_init = triangulate(cameras, pts2d)
err_init = reprojection_error(cameras, pts3d_init, pts2d)

# ... then refine the rig by fly-as-target bundle adjustment.
cameras_ba, ba_res = bundle_adjust_cameras(
    cameras, pts2d, conf, skeleton, **bundle_adjust_kwargs
)
pts3d_ba = triangulate(cameras_ba, pts2d)
err_ba = reprojection_error(cameras_ba, pts3d_ba, pts2d)

fi, fc = np.isfinite(err_init), np.isfinite(err_ba)
print(f"bundle adjustment: {ba_res.nfev} fn evals, final cost {ba_res.cost:.4g}")
print(
    f"median reproj error  before {np.median(err_init[fi]):.2f}px  ->  "
    f"after {np.median(err_ba[fc]):.2f}px"
)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(0, 40, 60)
axa.hist(
    err_init[fi],
    bins=bins,
    alpha=0.6,
    label=f"before (med {np.median(err_init[fi]):.1f})",
)
axa.hist(
    err_ba[fc], bins=bins, alpha=0.6, label=f"after (med {np.median(err_ba[fc]):.1f})"
)
axa.set_xlabel("reprojection error (px)")
axa.set_ylabel("count")
axa.legend()
axa.set_title("Reprojection error: nominal vs bundle-adjusted cameras")

shift = np.linalg.norm(
    np.array([c.position for c in cameras_ba])
    - np.array([c.position for c in cameras]),
    axis=1,
)
axb.bar(view_names, shift, color="tab:purple")
axb.set_ylabel("camera-center shift (world units)")
axb.set_title("How far bundle adjustment moved each camera")
fig.tight_layout()
plt.show()

## Step 11 — robust triangulation (RANSAC)

`pipeline.reconstruct_ransac` — the default reconstructor — builds each 3D point
from its *largest set of mutually consistent views* (RANSAC), so a badly
mislocated 2D detection never enters the fit (rather than corrupting it and being
trimmed afterward). Non-inlier observations are set to NaN. The result is the 3D
pose sequence `(T, 38, 3)`.

In [ ]:
pts3d, pts2d_clean, reproj = reconstruct_ransac(
    cameras_ba, pts2d, threshold=15.0, min_inliers=2
)
fin = np.isfinite(reproj)
n_tri = int(np.isfinite(pts3d).all(-1).sum())
print(f"3D points: {pts3d.shape};  triangulated {n_tri} of {pts3d[..., 0].size}")
print(
    f"reproj error (px): median {np.median(reproj[fin]):.2f}  "
    f"p95 {np.percentile(reproj[fin], 95):.2f}"
)

t = N_FRAMES // 2
fig = plt.figure(figsize=(11, 5))
for k, (elev, azim) in enumerate([(20, -60), (85, -90)]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    P, col = pts3d[t], point_colors_rgb(skeleton)
    for a, b in skeleton.bones:
        if np.isfinite(P[[a, b]]).all():
            ax.plot(*P[[a, b]].T, color=col[a])
    finite = np.isfinite(P).all(-1)
    ax.scatter(*P[finite].T, c=col[finite], s=10)
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(f"frame {t}  (elev={elev}, azim={azim})", fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")
fig.suptitle("Triangulated 3D pose (RANSAC multi-view consensus)")
fig.tight_layout()
plt.show()

## Step 12 — the same thing in one call, and save the result

Everything from Step 10 on — bundle-adjust → reconstruct — is exactly what
`pipeline.run_from_points2d` (and therefore `deeperfly run`) does internally. We
call it directly on the raw `pts2d` / `conf`, save the `PoseResult` to HDF5, and
reload it to confirm the round-trip.

In [ ]:
result = run_from_points2d(
    cameras,
    skeleton,
    pts2d,
    conf,
    do_bundle_adjust=True,
    bundle_adjust_kwargs=bundle_adjust_kwargs,
    triangulation="ransac",
    fps=FPS,
    meta={"source": str(DATA_DIR), "n_frames_input": N_FRAMES},
)

out = REPO / "results" / "fly_pose_walkthrough.h5"
out.parent.mkdir(parents=True, exist_ok=True)
result.save(out)

re = result.reproj_error
fr = np.isfinite(re)
print(f"wrote {out}")
print(f"  views x frames x points : {result.pts2d.shape}")
print(f"  3D points               : {result.pts3d.shape}")
print(
    f"  reprojection error (px) : median {np.median(re[fr]):.2f}  "
    f"p95 {np.percentile(re[fr], 95):.2f}"
)

reloaded = PoseResult.load(out)
print(
    f"  reloaded {reloaded.n_views} views x {reloaded.n_frames} frames; "
    f"has 3D = {reloaded.pts3d is not None}"
)

### Sanity check: reproject the bundle-adjusted 3D back onto every camera

If bundle adjustment and triangulation are consistent, projecting the recovered 3D pose
through the bundle-adjusted cameras should land on the fly in each raw view.

In [ ]:
t = 0
proj = np.asarray(result.cameras.project(result.pts3d[t]))  # (V, 38, 2)
col = point_colors_rgb(skeleton)
fig, axes = plt.subplots(2, 4, figsize=(12, 4))
for v, ax in enumerate(axes.ravel()):
    if v < len(view_names):
        ax.set_facecolor("k")
        ax.imshow(windows[view_src[view_names[v]]][t], cmap="gray")
        for a, b in skeleton.bones:
            if np.isfinite(proj[v, [a, b]]).all():
                ax.plot(proj[v, [a, b], 0], proj[v, [a, b], 1], color=col[a])
        ax.set_title(view_names[v], fontsize=8)
    ax.axis("off")
fig.suptitle("Bundle-adjusted 3D pose reprojected onto every camera (frame 0)")
fig.tight_layout()
plt.show()

## Mapping back to the CLI

| Notebook step | CLI / library call |
| --- | --- |
| Steps 1–3 | `Config.detection_plan`, `Config.camera_group`, `Config.skeleton`, `pose2d.stream.load_models` |
| Steps 4–8 | `inference.detect_sequence` (the streaming `deeperfly.detect_2d` wraps it) |
| Steps 9–12 | `pipeline.run_from_points2d` (bundle-adjust → reconstruct → save) |

`deeperfly run` drives all of this from one config. `deeperfly init` writes a
self-contained `config.toml` (the detection plan — `[[sources]]` /
`[[pose2d.preprocessors]]` / `[[pose2d.models]]` / `[[pose2d.pathways]]` with `[pose2d.output_points]`
mappings — plus the `[cameras.*]` rig, the `[skeleton]`, and the `[pipeline]`).
The default config's sources already map to this recording's `camera_0.mp4` …
`camera_6.mp4`, so the whole notebook collapses to:

```bash
deeperfly init config.toml          # then edit the plan/[cameras] for your own rig
deeperfly run examples/data -c config.toml -o out/   # detect -> 3D -> overlay videos

deeperfly inspect out/results.h5                       # inspect the saved result
```

`run` writes `out/results.h5` (the `PoseResult`) plus one MP4 per
`[[visualization.videos]]` entry — by default `out/pose2d.mp4` (the
raw 2D detections) and `out/pose3d.mp4` (the triangulated skeleton reprojected
into every view). Enabled stages reuse their cached outputs while their config is
unchanged, so editing the triangulation or the videos and re-running recomputes
only the affected stages — the slow 2D detection is reused. Visibility is now
intrinsic to the plan (a `(view, point)` no pathway writes stays `NaN`), and the
front camera is just one source feeding two pathways — the manual cells above
unfold exactly what `detect_sequence` does internally.